# Seaborn

## Note préliminaire sur les données

Les données sont volumineuses : 10,8 millions de transactions, plus de 4 Gio en mémoire pour le fichier entier. Nous n'en chargerons que les colonnes dont les exercices ont besoin, soit environ 1 Gio, et nous définirons une ou plusieurs fonctions par exercice pour créer nos diagrammes.

De cette manière, à la sortie de chacune des fonctions, les variables temporaires utilisées seront supprimées et la mémoire pourra être réutilisée.

## Imports

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time

## Chargement des données

Pour ces travaux pratiques, nous allons utiliser des données de transactions immobilières sur la France entière, de janvier 2014 à juin 2022 : l'année 2022 n'en couvre que six mois. Ces données ne couvrent ni l'Alsace ni la Moselle (départements 67, 68 et 57), ni Mayotte (976).

Ces données proviennent du site d'[open data français](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/). Nous utiliserons là une version retravaillée qui regroupe les transactions, qui provient du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF) sur GitHub.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf

Chargez les données avec `pd.read_parquet`, en ne lisant que les colonnes de la liste `colonnes_utiles` : le fichier entier occuperait plus de quatre fois plus de mémoire.

Créez ensuite une colonne `date` à partir des colonnes `annee_mutation`, `mois_mutation` et `jour_mutation` avec `pd.to_datetime`, puis supprimez ces trois colonnes pour économiser de la mémoire.

Enfin, affichez les types des colonnes de la `DataFrame` et sa taille en mémoire, chaînes de caractères comprises (l'index, `id_mutation`, en contient). Vous devez obtenir 10 839 063 lignes, 11 colonnes et « memory usage: 1.0 GB ».

**Documentation utile**

- [`pandas.read_parquet`](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`pandas.DataFrame.drop`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)
- [`pandas.DataFrame.info`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)

In [ ]:
colonnes_utiles = [
  "annee_mutation",
  "mois_mutation",
  "jour_mutation",
  "code_departement",
  "valeur_fonciere",
  "nombre_lots",
  "nombre_pieces_principales",
  "nombre_maisons",
  "nombre_appartements",
  "surface_reelle_bati_total",
  "surface_reelle_bati_maisons",
  "surface_reelle_bati_appartements",
  "surface_terrain_total",
]

# Votre code ici

### Conseils supplémentaires

`pd.read_parquet` reçoit la liste des colonnes à lire par son argument `columns`. `pd.to_datetime` accepte un dictionnaire dont les clés `year`, `month` et `day` donnent les colonnes de l'année, du mois et du jour. `df.drop(columns=[...])` renvoie une nouvelle `DataFrame`, sans ces colonnes : affectez-la à `df`. `df.info()` ne compte pas la place des chaînes de caractères (sa taille finit alors par « + ») ; `df.info(memory_usage="deep")` la mesure.

```python
df = pd.read_parquet("dvf/dvf-linearized-2014-2022.parquet", columns=...)
df["date"] = pd.to_datetime(dict(year=df.annee_mutation, month=..., day=...))
df = df.drop(columns=[...])
df.info(memory_usage=...)
```

### Solution

In [ ]:
df = pd.read_parquet("dvf/dvf-linearized-2014-2022.parquet", columns=colonnes_utiles)
df["date"] = pd.to_datetime(
  dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df = df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"])
df.info(memory_usage="deep")

## Nombre de transactions par année : agréger avant de tracer

Pour commencer, nous allons compter les transactions (lignes du jeu de données) par année, de deux façons, et comparer ce qu'elles coûtent.

1. Dans `decompte_sur_les_lignes`, tirez un échantillon d'un million de transactions avec `df.sample`, puis tracez leur décompte par année avec `sns.countplot`, à partir de l'attribut `dt.year` de la colonne `date` : seaborn reçoit toutes les lignes et les compte lui-même. Mesurez la durée du tracé avec `time.perf_counter`.
2. Dans `decompte_agrege`, comptez d'abord les transactions de chaque année, sur toutes les lignes, avec `value_counts`, puis tracez ces quelques nombres avec `sns.barplot`, et mesurez de même la durée. Donnez un titre à la figure et à ses axes ; `sns.despine` retire les bordures du haut et de droite.

Vous devez trouver 1 537 648 transactions en 2021. Combien de temps prendrait `countplot` sur les 10,8 millions de lignes ? Que remarquez-vous pour 2022 ? Expliquez-le à l'aide de la présentation des données, plus haut.

**Documentation utile**

- [`pandas.DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)
- [`time.perf_counter`](https://docs.python.org/fr/3/library/time.html#time.perf_counter)
- [`pandas.Series.dt.year`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html)
- [`pandas.Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
- [`seaborn.despine`](https://seaborn.pydata.org/generated/seaborn.despine.html)

In [ ]:
def decompte_sur_les_lignes() -> None:
  pass  # Votre code ici


def decompte_agrege() -> None:
  pass  # Votre code ici


decompte_sur_les_lignes()
decompte_agrege()

### Conseils supplémentaires

`df.sample(1_000_000, random_state=0)` tire un million de lignes au hasard, les mêmes à chaque exécution. `time.perf_counter()` renvoie un instant en secondes : la durée est la différence de deux appels. `sns.countplot` compte lui-même les lignes : passez-lui la série des années par `x`, et l'`Axes` où tracer par `ax`. `df.date.dt.year.value_counts()` renvoie une série indexée par les années, que `sort_index()` remet dans l'ordre ; `sns.barplot` reçoit alors son index par `x` et ses valeurs par `y`, une seule valeur par barre. `ax.set_title`, `ax.set_xlabel` et `ax.set_ylabel` nomment la figure et ses axes. Le `countplot` d'un million de lignes prend plusieurs secondes ; sur les 10,8 millions, il prendrait plus d'une minute, à chaque essai.

```python
def decompte_sur_les_lignes() -> None:
  echantillon = df.sample(..., random_state=0)
  debut = time.perf_counter()
  fig, ax = plt.subplots()
  sns.countplot(x=..., ax=ax)
  ax.set_title(...)
  plt.show()
  duree = time.perf_counter() - debut
  print(f"countplot sur un million de lignes : {duree:.1f} s")


def decompte_agrege() -> None:
  debut = time.perf_counter()
  decompte = df.date.dt.year.value_counts().sort_index()
  print(decompte)
  fig, ax = plt.subplots()
  sns.barplot(x=..., y=..., ax=ax)
  sns.despine(fig)
  ...
  plt.show()
  print(f"value_counts puis barplot : {time.perf_counter() - debut:.1f} s")
```

### Solution

In [ ]:
def decompte_sur_les_lignes() -> None:
  echantillon = df.sample(1_000_000, random_state=0)
  debut = time.perf_counter()
  fig, ax = plt.subplots()
  sns.countplot(x=echantillon.date.dt.year, ax=ax)
  ax.set_title("Transactions par année, sur un échantillon d'un million")
  plt.show()
  duree = time.perf_counter() - debut
  print(f"countplot sur un million de lignes : {duree:.1f} s")
  # La durée croît avec le nombre de lignes : environ dix fois plus pour toutes
  print(f"soit environ {duree * len(df) / 1_000_000:.0f} s pour les {len(df)} lignes")


def decompte_agrege() -> None:
  debut = time.perf_counter()
  # 10,8 millions de lignes réduites à 9 nombres avant le tracé
  decompte = df.date.dt.year.value_counts().sort_index()
  print(decompte)
  fig, ax = plt.subplots()
  sns.barplot(x=decompte.index, y=decompte.values, ax=ax)
  sns.despine(fig)
  ax.set_title("Décompte des transactions par année")
  ax.set_xlabel("Année")
  ax.set_ylabel("Nombre de transactions")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(places=1))
  # 2022 ne couvre que janvier à juin : sa barre ne se compare pas aux autres
  plt.show()
  print(f"value_counts puis barplot : {time.perf_counter() - debut:.1f} s")


decompte_sur_les_lignes()
decompte_agrege()

## Valeur foncière totale par département

Dans cet exercice, nous allons regrouper nos données selon les valeurs d'une colonne (ici le département de la transaction) et afficher la valeur foncière totale échangée par groupe obtenu.

- Regroupez les données par département (`code_departement`) avec `groupby`, et calculez la somme de `valeur_fonciere` de chaque groupe.
- Affichez ces sommes avec `sns.barplot`, du département qui en a le plus à celui qui en a le moins.

In [ ]:
def valeur_par_departement() -> None:
  pass  # Votre code ici


valeur_par_departement()

### Conseils supplémentaires

`code_departement` est une colonne catégorielle : `groupby(..., observed=True)` ne garde que les départements présents dans les données. `sort_values(ascending=False)` trie les sommes de la plus grande à la plus petite, et l'argument `order` de `sns.barplot` donne l'ordre des barres. Avec une série, `x` reçoit son index (les départements) et `y` ses valeurs ; `figsize=(30, 10)` élargit la figure pour que les 97 départements se lisent.

```python
def valeur_par_departement() -> None:
  valeurs = (
    df.groupby(df.code_departement, observed=True)
    .valeur_fonciere.sum()
    .sort_values(...)
  )

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=valeurs.index, y=valeurs, order=..., ax=ax)
  ...
  plt.show()
```

### Solution

In [ ]:
def valeur_par_departement() -> None:
  valeurs = (
    df.groupby(df.code_departement, observed=True)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
  )

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=valeurs.index, y=valeurs, order=valeurs.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Montant total des transactions par département")
  ax.set_xlabel("Département")
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  plt.show()


valeur_par_departement()

## Valeur foncière totale par région et par an

Nous allons étudier dans cette partie comment décliner une variable catégorielle en fonction d'une autre variable : nous allons produire un diagramme de la valeur foncière totale échangée par région, et cette valeur sera elle-même déclinée par année.

Pour cela, commençons par récupérer un jeu de données qui nous permettra de regrouper les départements en régions :

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv -O regions.csv

La fonction suivante renvoie une `DataFrame` qui permettra de joindre facilement la région à notre `DataFrame` de travail : son index est le code du département, écrit comme dans nos données (« 1 », et non « 01 »), et sa colonne `nom_region` donne le nom de la région.

In [ ]:
def lire_regions() -> pd.DataFrame:
  regions = pd.read_csv(
    "regions.csv",
    header=0,
    names=["code_departement", "nom_departement", "nom_region"],
    usecols=["code_departement", "nom_region"],
    index_col="code_departement",
    dtype=dict(code_departement=str, nom_region="category"),
  )
  # DVF écrit les départements sans zéro initial (« 1 », et non « 01 »)
  return regions.rename(index=lambda code: code.lstrip("0"))

- Ajoutez le nom de la région à chaque transaction avec `DataFrame.join`, puis vérifiez que la colonne `nom_region` n'a aucune valeur manquante (`isna().sum()` vaut 0) : `groupby` écarterait sans rien dire une transaction sans région.
- Regroupez les données par région puis par année avec `groupby`, et calculez la somme de `valeur_fonciere` de chaque groupe.
- Affichez ces sommes avec `sns.barplot`, une barre par année dans chaque région (argument `hue`). Le tutoriel de seaborn traite de ces données catégorielles déclinées par une autre variable.

**Documentation utile**

- [`pandas.DataFrame.join`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html)
- [Tutoriel de seaborn : données catégorielles](https://seaborn.pydata.org/tutorial/introduction.html#plots-for-categorical-data)

In [ ]:
def valeur_par_region_et_annee() -> None:
  pass  # Votre code ici


valeur_par_region_et_annee()

### Conseils supplémentaires

`df.join(lire_regions(), on="code_departement")` associe à chaque transaction la ligne de `lire_regions()` dont l'index vaut son `code_departement`. `groupby` prend une liste de clés : la colonne `nom_region` et la série `date.dt.year`, renommée `annee` par `rename`. `reset_index()` refait de ces clés des colonnes, ce qu'attend `sns.barplot(donnees, x=..., y=..., hue=...)`. L'année est un entier : seaborn la traiterait comme une variable continue (une palette en dégradé, une légende qui saute des années) ; `astype(str)` en fait une catégorie. `ax.tick_params(axis="x", rotation=45)` incline les noms des régions, et `ax.get_legend().set_title(...)` titre la légende.

```python
def valeur_par_region_et_annee() -> None:
  avec_regions = df.join(lire_regions(), on="code_departement")
  print("Transactions sans région :", ...)
  ordre_regions = (
    avec_regions.groupby(avec_regions.nom_region, observed=True)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .index
  )
  donnees = (
    avec_regions.groupby(
      [avec_regions.nom_region, avec_regions.date.dt.year.rename("annee")],
      observed=True,
    )
    .valeur_fonciere.sum()
    .reset_index()
  )
  donnees["annee"] = ...

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(donnees, x=..., y=..., hue=..., order=ordre_regions, ax=ax)
  ...
  plt.show()
```

### Solution

In [ ]:
def valeur_par_region_et_annee() -> None:
  avec_regions = df.join(lire_regions(), on="code_departement")
  print("Transactions sans région :", avec_regions.nom_region.isna().sum())
  ordre_regions = (
    avec_regions.groupby(avec_regions.nom_region, observed=True)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .index
  )
  donnees = (
    avec_regions.groupby(
      [avec_regions.nom_region, avec_regions.date.dt.year.rename("annee")],
      observed=True,
    )
    .valeur_fonciere.sum()
    .reset_index()
  )
  donnees["annee"] = donnees.annee.astype(str)

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(
    donnees,
    x="nom_region",
    y="valeur_fonciere",
    hue="annee",
    order=ordre_regions,
    ax=ax,
  )
  sns.despine(fig)
  ax.set_title("Montant total des transactions par région et par année")
  ax.set_xlabel("Région")
  ax.tick_params(axis="x", rotation=45)
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  ax.get_legend().set_title("Année")
  plt.show()


valeur_par_region_et_annee()

## Corrélations de Pearson et de Spearman

La corrélation de Pearson mesure une relation linéaire. Celle de Spearman, la corrélation de Pearson des rangs, mesure une relation monotone (croissante ou décroissante), sans tenir compte des écarts entre les valeurs. Nous allons les comparer sur les transactions de 2021, entre la valeur foncière, les surfaces et le nombre de pièces (la liste `variables`).

- Calculez les deux matrices de corrélation de ces variables avec `DataFrame.corr` et son argument `method`.
- Affichez-les côte à côte, sur les deux `Axes` d'une même figure, avec `sns.heatmap`, chaque case annotée de sa valeur.
- Comparez la ligne de `valeur_fonciere` dans les deux matrices : avec `surface_reelle_bati_total`, vous devez trouver 0,10 (Pearson) et 0,55 (Spearman).
- Affichez les cinq valeurs foncières les plus élevées avec `nlargest`, puis recalculez la corrélation de Pearson entre `valeur_fonciere` et `surface_reelle_bati_total` sans le 1 % de transactions les plus chères (`quantile`).

Qu'est-ce qui explique l'écart entre les deux matrices sur la ligne de `valeur_fonciere` ?

**Documentation utile**

- [`pandas.DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)
- [`pandas.Series.nlargest`](https://pandas.pydata.org/docs/reference/api/pandas.Series.nlargest.html)
- [`pandas.Series.quantile`](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html)
- [`pandas.Series.corr`](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html)

In [ ]:
variables = [
  "valeur_fonciere",
  "surface_reelle_bati_total",
  "surface_reelle_bati_maisons",
  "surface_reelle_bati_appartements",
  "surface_terrain_total",
  "nombre_pieces_principales",
]


def correlations() -> None:
  pass  # Votre code ici


correlations()

### Conseils supplémentaires

`df.loc[df.date.dt.year == 2021, variables]` garde les transactions de 2021 et les colonnes de la liste. `corr(method="spearman")` calcule la corrélation de Spearman (`"pearson"` par défaut). `fig, axes = plt.subplots(1, 2, figsize=(18, 7))` crée une figure de deux `Axes`, `axes[0]` et `axes[1]`, que `sns.heatmap` reçoit par `ax`. `annot=True` écrit la valeur dans chaque case, `fmt=".2f"` avec deux décimales ; `vmin=-1, vmax=1` fixent l'échelle des couleurs, et une palette divergente (`cmap="vlag"`) en distingue le signe. `quantile(0.99)` renvoie la valeur sous laquelle tombent 99 % des transactions, et `serie.corr(autre_serie)` la corrélation de deux colonnes.

```python
def correlations() -> None:
  ventes_2021 = df.loc[df.date.dt.year == 2021, variables]
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, methode in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      ventes_2021.corr(method=...),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=...,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Corrélations ({methode}, 2021)")
  fig.tight_layout()
  plt.show()

  print(ventes_2021.valeur_fonciere.nlargest(...))
  seuil = ventes_2021.valeur_fonciere.quantile(...)
  sans_extremes = ventes_2021[ventes_2021.valeur_fonciere <= seuil]
  ...
```

### Solution

In [ ]:
def correlations() -> None:
  ventes_2021 = df.loc[df.date.dt.year == 2021, variables]
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, methode in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      ventes_2021.corr(method=methode),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=True,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Corrélations ({methode}, 2021)")
  fig.tight_layout()
  plt.show()

  print(ventes_2021.valeur_fonciere.nlargest(5))
  seuil = ventes_2021.valeur_fonciere.quantile(0.99)
  sans_extremes = ventes_2021[ventes_2021.valeur_fonciere <= seuil]
  pearson = sans_extremes.valeur_fonciere.corr(sans_extremes.surface_reelle_bati_total)
  print(
    f"Pearson sans le 1 % le plus cher (au-dessus de {seuil:.0f} €) : {pearson:.2f}"
  )
  # Quelques ventes énormes (jusqu'à 620 millions d'euros : des immeubles, des
  # ensembles de lots) dominent la variance de valeur_fonciere, donc Pearson, qui
  # tombe à 0,10. Spearman ne voit que leur rang : 0,55. Sans le 1 % le plus cher,
  # Pearson remonte à 0,37. Entre la surface bâtie totale et le nombre de pièces,
  # sans valeurs aussi extrêmes, les deux s'accordent (0,92 et 0,96).


correlations()

## Prix des maisons et des appartements

Dans cet exercice, nous allons calculer le prix au mètre carré des appartements et des maisons séparément, et afficher son évolution mois par mois.

- Pour les appartements, ne gardez que les lignes de la `DataFrame` où :
    - `nombre_appartements`, `valeur_fonciere` et `surface_reelle_bati_appartements` sont strictement positifs ;
    - `nombre_maisons` est nul.
- De la même manière, pour les maisons, ne gardez que les lignes où :
    - `nombre_maisons`, `valeur_fonciere` et `surface_reelle_bati_maisons` sont strictement positifs ;
    - `nombre_appartements` est nul.
- Calculez le prix au mètre carré de chaque transaction gardée (`valeur_fonciere` divisée par la surface), puis la médiane de ces prix pour chaque mois. Les grosses ventes (un immeuble entier, des dizaines de lots) tireraient une moyenne ou un rapport de sommes, pas la médiane.
- Affichez les deux courbes sur une même figure avec `sns.lineplot`, une couleur par type de bien (argument `hue`), l'axe vertical titré « Prix médian au m² ».

Pour les appartements, en juin 2022, vous devez trouver 3 723 €/m² (2 051 €/m² pour les maisons).

**Documentation utile**

- [`pandas.Series.dt.to_period`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.to_period.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [`pandas.DataFrame.melt`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html)
- [`seaborn.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

In [ ]:
def prix_maisons_appartements() -> None:
  pass  # Votre code ici


prix_maisons_appartements()

### Conseils supplémentaires

Un masque combine ses conditions par `&`, chacune entre parenthèses : `(df.nombre_maisons > 0) & (df.nombre_appartements == 0)`. `date.dt.to_period("M")` ramène chaque date à son mois, et `.dt.to_timestamp()` en refait une date (le premier du mois), que `sns.lineplot` place sur un axe de temps ; `prix_m2.groupby(mois).median()` donne alors la médiane de chaque mois. `sns.lineplot` attend une seule `DataFrame` au format long, une ligne par mois et par type de bien : `pd.concat([...], axis=1)` met les deux séries côte à côte (chacune nommée par `rename`), `reset_index()` fait du mois une colonne, et `melt` empile les deux colonnes de prix en une seule, à côté d'une colonne qui dit le type de bien.

```python
def prix_maisons_appartements() -> None:
  masque_appartements = (df.nombre_appartements > 0) & (df.nombre_maisons == 0) & ...
  masque_maisons = ...

  def prix_m2_median(masque: pd.Series, colonne_surface: str) -> pd.Series:
    ventes = df[masque]
    prix_m2 = ventes.valeur_fonciere / ventes[colonne_surface]
    mois = ventes.date.dt.to_period("M").dt.to_timestamp()
    return prix_m2.groupby(mois).median()

  donnees = pd.concat(
    [
      prix_m2_median(masque_maisons, ...).rename("maisons"),
      prix_m2_median(masque_appartements, ...).rename("appartements"),
    ],
    axis=1,
  )
  donnees = donnees.reset_index().melt(
    id_vars="date", var_name="type_de_bien", value_name="prix_m2"
  )

  fig, ax = plt.subplots()
  sns.lineplot(donnees, x=..., y=..., hue=..., ax=ax)
  ...
  plt.show()
```

### Solution

In [ ]:
def prix_maisons_appartements() -> None:
  masque_appartements = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  masque_maisons = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )

  def prix_m2_median(masque: pd.Series, colonne_surface: str) -> pd.Series:
    ventes = df[masque]
    prix_m2 = ventes.valeur_fonciere / ventes[colonne_surface]
    mois = ventes.date.dt.to_period("M").dt.to_timestamp()
    return prix_m2.groupby(mois).median()

  donnees = pd.concat(
    [
      prix_m2_median(masque_maisons, "surface_reelle_bati_maisons").rename("maisons"),
      prix_m2_median(masque_appartements, "surface_reelle_bati_appartements").rename(
        "appartements"
      ),
    ],
    axis=1,
  )
  donnees = donnees.reset_index().melt(
    id_vars="date", var_name="type_de_bien", value_name="prix_m2"
  )

  fig, ax = plt.subplots()
  sns.lineplot(donnees, x="date", y="prix_m2", hue="type_de_bien", ax=ax)
  sns.despine(fig)
  ax.tick_params(axis="x", rotation=45)
  ax.set_xlabel("Mois")
  ax.set_ylabel("Prix médian au m²")
  ax.set_title("Prix médian au mètre carré")
  ax.set_ylim(bottom=0)
  ax.get_yaxis().set_major_formatter(
    matplotlib.ticker.EngFormatter(unit="€/m²", places=1)
  )
  ax.get_legend().set_title("Type de bien")
  plt.show()


prix_maisons_appartements()

## Appartements vendus en 2021 (facultatif)

Cet exercice facultatif trace trois diagrammes que les exercices précédents n'emploient pas : l'histogramme, le diagramme en violons et le nuage de points. Il porte sur les ventes d'un seul appartement en 2021 : `nombre_appartements` vaut 1, `nombre_maisons` est nul, `valeur_fonciere` et `surface_reelle_bati_appartements` sont strictement positifs.

1. Sélectionnez ces ventes, ajoutez-leur la région (`lire_regions`, comme plus haut) et une colonne `prix_m2`, la valeur foncière divisée par la surface. Vous devez en trouver 418 559, au prix médian de 162 000 €.
2. Tracez l'histogramme de `valeur_fonciere` avec `sns.histplot`, en échelle logarithmique. Que montre-t-il qu'un histogramme en échelle linéaire écraserait ?
3. Tracez la répartition du prix au m² de chaque région avec `sns.violinplot`, une région par ligne, de la plus chère à la moins chère (prix médian), sur les ventes dont le prix au m² est compris entre le 1er et le 99e centile. Que montre le violon de l'Île-de-France ?
4. Tracez la valeur foncière en fonction de la surface avec `sns.scatterplot`, sur un échantillon de 5 000 ventes, les points transparents et les deux axes en échelle logarithmique. Pourquoi un échantillon ?

**Documentation utile**

- [`seaborn.histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html)
- [`pandas.Series.between`](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html)
- [`seaborn.scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html)

In [ ]:
def appartements_2021() -> None:
  pass  # Votre code ici


appartements_2021()

### Conseils supplémentaires

Le masque suit celui de l'exercice précédent, avec `df.date.dt.year == 2021` et `df.nombre_appartements == 1`. `log_scale=True` met l'axe de `sns.histplot` en échelle logarithmique, et ses classes avec. `quantile([0.01, 0.99])` renvoie les deux centiles, et `serie.between(bas, haut)` garde les valeurs comprises entre eux. Avec `x="prix_m2", y="nom_region"`, `sns.violinplot` dessine un violon par région, couché ; `order` reçoit les régions triées par prix médian, et `cut=0` arrête chaque violon à ses données. `sample(5_000, random_state=0)` tire l'échantillon, `alpha=0.3` rend les points transparents, et `ax.set(xscale="log", yscale="log")` passe les deux axes en échelle logarithmique.

```python
def appartements_2021() -> None:
  masque = (df.date.dt.year == 2021) & (df.nombre_appartements == 1) & ...
  appartements = df[masque].join(lire_regions(), on="code_departement")
  appartements["prix_m2"] = ...
  print(len(appartements), appartements.valeur_fonciere.median())

  fig, ax = plt.subplots()
  sns.histplot(appartements, x="valeur_fonciere", log_scale=..., ax=ax)
  ...
  plt.show()

  bas, haut = appartements.prix_m2.quantile([0.01, 0.99])
  plausibles = appartements[appartements.prix_m2.between(bas, haut)]
  ordre = (
    plausibles.groupby("nom_region", observed=True)
    .prix_m2.median()
    .sort_values(ascending=False)
    .index
  )
  fig, ax = plt.subplots(figsize=(10, 8))
  sns.violinplot(plausibles, x=..., y=..., order=ordre, cut=0, ax=ax)
  ...
  plt.show()

  echantillon = appartements.sample(..., random_state=0)
  fig, ax = plt.subplots()
  sns.scatterplot(echantillon, x=..., y=..., alpha=0.3, ax=ax)
  ax.set(xscale="log", yscale="log")
  ...
  plt.show()
```

### Solution

In [ ]:
def appartements_2021() -> None:
  masque = (
    (df.date.dt.year == 2021)
    & (df.nombre_appartements == 1)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  appartements = df[masque].join(lire_regions(), on="code_departement")
  appartements["prix_m2"] = (
    appartements.valeur_fonciere / appartements.surface_reelle_bati_appartements
  )
  print(len(appartements), appartements.valeur_fonciere.median())

  fig, ax = plt.subplots()
  sns.histplot(appartements, x="valeur_fonciere", log_scale=True, ax=ax)
  sns.despine(fig)
  ax.set_title("Prix des appartements vendus en 2021")
  ax.set_xlabel("Valeur foncière (€, échelle logarithmique)")
  ax.set_ylabel("Nombre de ventes")
  # La moitié des ventes entre 102 000 et 255 000 € (les quartiles), et une traîne
  # de ventes à moins de 1 000 € (447), qu'en échelle linéaire la vente à près de
  # 90 millions écraserait contre l'axe
  plt.show()

  bas, haut = appartements.prix_m2.quantile([0.01, 0.99])
  plausibles = appartements[appartements.prix_m2.between(bas, haut)]
  ordre = (
    plausibles.groupby("nom_region", observed=True)
    .prix_m2.median()
    .sort_values(ascending=False)
    .index
  )
  fig, ax = plt.subplots(figsize=(10, 8))
  sns.violinplot(plausibles, x="prix_m2", y="nom_region", order=ordre, cut=0, ax=ax)
  sns.despine(fig)
  ax.set_title("Prix au m² des appartements vendus en 2021, par région")
  ax.set_xlabel("Prix au m² (€)")
  ax.set_ylabel("Région")
  # L'Île-de-France a le violon le plus étalé, jusqu'à plus de 10 000 €/m² :
  # Paris et sa banlieue n'ont pas le même marché
  plt.show()

  # Les 418 559 points se recouvriraient en une tache, longue à tracer
  echantillon = appartements.sample(5_000, random_state=0)
  fig, ax = plt.subplots()
  sns.scatterplot(
    echantillon,
    x="surface_reelle_bati_appartements",
    y="valeur_fonciere",
    alpha=0.3,
    ax=ax,
  )
  ax.set(xscale="log", yscale="log")
  sns.despine(fig)
  ax.set_title("Valeur foncière et surface (5 000 ventes de 2021)")
  ax.set_xlabel("Surface (m²)")
  ax.set_ylabel("Valeur foncière (€)")
  plt.show()


appartements_2021()